In [ ]:
# Cell 1 — Install missing Colab packages (always first, no exceptions)
!pip install -r requirements-colab.txt -q

In [ ]:
# Cell 2 — Mount Google Drive
# Dataset at: MyDrive/aerial_detection/classification_dataset/
# Upload once — survives session restarts without re-uploading 200MB+
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3 — Imports + Seeds
# stdlib
import json
import os
import random
from datetime import datetime, timezone
from pathlib import Path

# third-party
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input

# Seeds — must match RANDOM_STATE used everywhere in project
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

print(f"TensorFlow: {tf.__version__}")
print(f"NumPy:      {np.__version__}")

In [ ]:
# Cell 4 — Config (all constants here, never inline)
RANDOM_STATE  = 42
IMAGE_SIZE    = (224, 224)
SAMPLE_N_GRID = 16     # 4×4 image grid per class
PIXEL_SAMPLE  = 500    # stratified: 250 bird + 250 drone for pixel stats
DRIFT_SAMPLE  = 200    # 100 bird + 100 drone for KS drift baseline

SPLITS = ("train", "valid", "test")
LABELS = ("bird", "drone")

DRIVE_ROOT  = Path("/content/drive/MyDrive/aerial_detection")
DATASET_DIR = DRIVE_ROOT / "classification_dataset"
EDA_OUT     = Path("data/eda")
CHARTS_DIR  = EDA_OUT / "charts"

EDA_OUT.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset dir: {DATASET_DIR}")
print(f"EDA output:  {EDA_OUT.resolve()}")
assert DATASET_DIR.exists(), f"Dataset not found at {DATASET_DIR} — check Drive mount"

In [ ]:
# Cell 5 — Class Distribution + Imbalance Decision

def get_paths(split: str, label: str) -> list:
    """Return sorted, deduplicated image paths for a given split and label."""
    folder = DATASET_DIR / split / label
    seen = set()
    paths = []
    for ext in ("*.jpg", "*.jpeg", "*.JPG", "*.JPEG", "*.png", "*.PNG"):
        for p in folder.glob(ext):
            # Dedup by lowercase name — macOS filesystem is case-insensitive;
            # *.jpg and *.JPG would both match the same file there.
            # Colab (Linux) is case-sensitive so this is a no-op there,
            # but costs nothing and prevents a confusing bug if run locally.
            if p.name.lower() not in seen:
                seen.add(p.name.lower())
                paths.append(p)
    return sorted(paths)


# ── Count images per (split, label) ──────────────────────────────────────────
counts = {
    split: {label: len(get_paths(split, label)) for label in LABELS}
    for split in SPLITS
}

df_counts = pd.DataFrame(counts).T
df_counts["total"] = df_counts.sum(axis=1)
print(df_counts.to_string())
print(f"\nGrand total: {df_counts['total'].sum()} images")


# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
x = range(len(SPLITS))
width = 0.35
ax.bar([i - width/2 for i in x], [counts[s]["bird"]  for s in SPLITS], width, label="bird",  color="steelblue")
ax.bar([i + width/2 for i in x], [counts[s]["drone"] for s in SPLITS], width, label="drone", color="darkorange")
ax.set_xticks(list(x))
ax.set_xticklabels(SPLITS)
ax.set_ylabel("Image count")
ax.set_title("Class Distribution per Split")
ax.legend()
plt.tight_layout()
plt.savefig(CHARTS_DIR / "class_distribution.png", dpi=100)
plt.show()
print(f"Saved: {CHARTS_DIR / 'class_distribution.png'}")


# ── Class weight calculation (sklearn 'balanced' formula) ─────────────────────
bird_train  = counts["train"]["bird"]    # 1414
drone_train = counts["train"]["drone"]   # 1248
total_train = bird_train + drone_train   # 2662

# Formula: total / (n_classes * class_count)
class_weight = {
    0: total_train / (2 * bird_train),    # ≈ 0.941
    1: total_train / (2 * drone_train),   # ≈ 1.066
}
# Both deviate from 1.0 — correct behaviour.
# class_weight[1] > class_weight[0] → drone misses are penalised more (security requirement).
# Neither class is exactly 1.0 — this matches sklearn compute_class_weight('balanced') exactly.


# ── Imbalance decision ────────────────────────────────────────────────────────
ratio = bird_train / drone_train

if ratio < 1.5:
    imbalance_decision = "mild_imbalance_use_class_weight"
else:
    imbalance_decision = "severe_imbalance_use_oversampling"

print(f"\nImbalance ratio: {ratio:.4f}")
print(f"Decision:        {imbalance_decision}")
print(f"class_weight:    {{{0}: {class_weight[0]:.4f}, {1}: {class_weight[1]:.4f}}}")

In [ ]:
# Cell 6 — Sample Image Grid (4×4 per class)

def save_image_grid(paths: list, title: str, save_path: Path) -> None:
    """Plot a 4×4 grid of images from the given paths."""
    sample = random.sample(paths, min(SAMPLE_N_GRID, len(paths)))
    fig, axes = plt.subplots(4, 4, figsize=(10, 10))
    fig.suptitle(title, fontsize=14)
    for ax, img_path in zip(axes.ravel(), sample):
        img = Image.open(img_path).convert("RGB").resize(IMAGE_SIZE)
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(img_path.name[:20], fontsize=6)
    plt.tight_layout()
    plt.savefig(save_path, dpi=100)
    plt.show()
    print(f"Saved: {save_path}")


save_image_grid(
    get_paths("train", "bird"),
    "Sample: Bird (Training Set)",
    CHARTS_DIR / "sample_grid_bird.png",
)
save_image_grid(
    get_paths("train", "drone"),
    "Sample: Drone (Training Set)",
    CHARTS_DIR / "sample_grid_drone.png",
)

In [ ]:
# Cell 7a — Load 500-Image Stratified Sample (slow step — isolated to avoid re-running)
# Keep bird and drone stacks separate so Cell 8 can overlay them without index slicing.

sample_bird  = random.sample(get_paths("train", "bird"),  250)
sample_drone = random.sample(get_paths("train", "drone"), 250)


def load_for_stats(img_path: Path) -> np.ndarray:
    """Load image, resize, divide by 255 for raw pixel distribution analysis.

    NOTE: This is for statistical analysis only, NOT for feeding a model.
    EfficientNetB0 uses preprocess_input() instead — see Cell 10.
    """
    img = Image.open(img_path).convert("RGB").resize(IMAGE_SIZE)
    return np.array(img, dtype=np.float32) / 255.0


print("Loading bird sample  ...", end=" ")
pixel_stack_bird  = np.stack([load_for_stats(p) for p in sample_bird])   # (250,224,224,3)
print(f"done  shape={pixel_stack_bird.shape}")

print("Loading drone sample ...", end=" ")
pixel_stack_drone = np.stack([load_for_stats(p) for p in sample_drone])  # (250,224,224,3)
print(f"done  shape={pixel_stack_drone.shape}")

pixel_stack = np.concatenate([pixel_stack_bird, pixel_stack_drone])       # (500,224,224,3)
print(f"Combined stack: {pixel_stack.shape}  ({pixel_stack.nbytes / 1e6:.0f} MB)")

In [ ]:
# Cell 7b — Compute and Print Pixel Statistics

mean_rgb = pixel_stack.mean(axis=(0, 1, 2))   # (3,) — per channel, across all pixels and images
std_rgb  = pixel_stack.std(axis=(0, 1, 2))    # (3,)

print("Pixel statistics (500-image stratified sample, /255 normalised):")
print(f"  mean_rgb: {mean_rgb.round(4)}")
print(f"  std_rgb:  {std_rgb.round(4)}")
print()
print("ImageNet reference:")
print(f"  mean_rgb: [0.485, 0.456, 0.406]")
print(f"  std_rgb:  [0.229, 0.224, 0.225]")
print()

# Normalization decision
# Custom CNN (Section 3): divide by 255.0
# EfficientNetB0 (Section 4 + drift baseline): preprocess_input() handles scaling internally
# We do NOT subtract ImageNet mean/std — EfficientNetB0's built-in layer handles that
print("Normalization decision:")
print("  Custom CNN      → divide by 255.0 only")
print("  EfficientNetB0  → preprocess_input() (scales to [-1, 1] internally)")

# Sanity check ranges
for i, ch in enumerate(["R", "G", "B"]):
    assert 0.3 <= mean_rgb[i] <= 0.7, f"mean_rgb[{ch}]={mean_rgb[i]:.4f} out of [0.3, 0.7] — preprocessing bug?"
    assert 0.1 <= std_rgb[i]  <= 0.35, f"std_rgb[{ch}]={std_rgb[i]:.4f} out of [0.1, 0.35] — preprocessing bug?"
print("\nRange checks passed.")

In [ ]:
# Cell 8 — RGB Channel Histograms (bird vs drone overlaid)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("RGB Channel Pixel Distribution — Bird vs Drone (training sample)", fontsize=13)

for i, channel in enumerate(["R", "G", "B"]):
    # ravel() returns a view when possible — no copy; faster than flatten() for 12M values
    bird_vals  = pixel_stack_bird[..., i].ravel()    # ~250 × 224 × 224 ≈ 12.5M values
    drone_vals = pixel_stack_drone[..., i].ravel()

    axes[i].hist(bird_vals,  bins=50, alpha=0.6, color="steelblue",  label="bird",  density=True)
    axes[i].hist(drone_vals, bins=50, alpha=0.6, color="darkorange", label="drone", density=True)
    axes[i].set_title(f"Channel {channel}")
    axes[i].set_xlabel("Pixel value (normalised)")
    axes[i].set_ylabel("Density")
    axes[i].legend()

plt.tight_layout()
plt.savefig(CHARTS_DIR / "pixel_histograms.png", dpi=100)
plt.show()
print(f"Saved: {CHARTS_DIR / 'pixel_histograms.png'}")

In [ ]:
# Cell 9 — Augmentation Preview + Lock Parameters
#
# These parameters are the SINGLE source of truth for augmentation in this project.
# Section 3 (data_loader.py) and Section 4 (transfer_learning.py) read from
# eda_stats.json["augmentation_config"] — never re-define them inline.
#
# Note: brightness_range works with flow_from_directory() but NOT with flow() on arrays.
# Section 3's data_loader.py must use flow_from_directory. If arrays are needed,
# replace brightness_range with tf.keras.layers.RandomBrightness as a preprocessing layer.

AUGMENTATION_CONFIG = {
    "rotation_range":     20,
    "zoom_range":         0.15,
    "width_shift_range":  0.1,
    "height_shift_range": 0.1,   # aerial objects vary vertically as well as horizontally
    "horizontal_flip":    True,
    "brightness_range":   [0.8, 1.2],
    "fill_mode":          "nearest",
}

from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Pick a drone image — more visually interesting for the preview
demo_path = random.choice(get_paths("train", "drone"))
demo_img  = Image.open(demo_path).convert("RGB").resize(IMAGE_SIZE)
demo_arr  = np.array(demo_img, dtype=np.float32) / 255.0  # normalise for display
demo_batch = demo_arr[np.newaxis]                          # (1, 224, 224, 3)

# Each augmentation applied independently for a clean comparison
aug_params = [
    ("Original",              {}),
    ("Rotation",              {"rotation_range": 20}),
    ("Zoom",                  {"zoom_range": 0.15}),
    ("H-Shift",               {"width_shift_range": 0.1}),
    ("V-Shift",               {"height_shift_range": 0.1}),
    ("H-Flip",                {"horizontal_flip": True}),
    ("Brightness",            {"brightness_range": [0.8, 1.2]}),
]

fig, axes = plt.subplots(1, 7, figsize=(21, 3))
fig.suptitle("Augmentation Preview — Drone Image", fontsize=12)

for ax, (label, params) in zip(axes, aug_params):
    if not params:
        aug_img = demo_arr
    else:
        gen = ImageDataGenerator(**{**params, "fill_mode": "nearest"})
        aug_img = next(gen.flow(demo_batch, batch_size=1))[0]
        aug_img = np.clip(aug_img, 0, 1)
    ax.imshow(aug_img)
    ax.set_title(label, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.savefig(CHARTS_DIR / "augmentation_preview.png", dpi=100)
plt.show()
print(f"Saved: {CHARTS_DIR / 'augmentation_preview.png'}")
print(f"Augmentation config: {AUGMENTATION_CONFIG}")

In [ ]:
# Cell 10 — Drift Baseline Embeddings
#
# CRITICAL PREPROCESSING NOTE:
# EfficientNetB0's preprocess_input() expects pixels in [0, 255] and scales to [-1, 1] internally.
# Dividing by 255 BEFORE preprocess_input() yields values in ~[-1/255, 1/255] — garbage embeddings.
# The preprocessing here MUST match exactly what Section 4 uses during training.
#
# Why save here (not compute in FastAPI):
#   1. 200-image EfficientNetB0 inference adds 30-60s to cold start — unacceptable for prod API
#   2. Baseline must be frozen to the original training distribution; recomputing later
#      loses that historical anchor. FastAPI just loads a numpy file (milliseconds).

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

# Load frozen base — pooling='avg' gives (None, 1280) output without extra GlobalAvgPool
base = EfficientNetB0(include_top=False, weights="imagenet", pooling="avg")
base.trainable = False

# Confirm weights loaded — wrong count means the download failed
n_params = base.count_params()
print(f"EfficientNetB0 base params: {n_params:,}")
assert 4_000_000 < n_params < 4_200_000, f"Unexpected param count {n_params} — check weight loading"


def load_for_efficientnet(img_path) -> np.ndarray:
    """Load and preprocess for EfficientNetB0 — do NOT divide by 255."""
    img = Image.open(img_path).convert("RGB").resize(IMAGE_SIZE)
    arr = np.array(img, dtype=np.float32)
    return preprocess_input(arr)   # scales to [-1, 1] internally


# 100 bird + 100 drone, fixed order: first 100 = bird (label 0), last 100 = drone (label 1)
baseline_bird  = random.sample(get_paths("train", "bird"),  100)
baseline_drone = random.sample(get_paths("train", "drone"), 100)
all_paths      = baseline_bird + baseline_drone

# 200 images × 224×224×3 × float32 ≈ 120MB — safe for Colab RAM (12GB limit)
# For larger baselines, use base.predict(generator, steps=N) instead
print("Running base.predict() on 200 images ...")
batch      = np.stack([load_for_efficientnet(p) for p in all_paths])
embeddings = base.predict(batch, verbose=1)   # shape (200, 1280)

labels = np.array([0] * 100 + [1] * 100, dtype=np.int32)

np.save(EDA_OUT / "reference_embeddings.npy", embeddings)
np.save(EDA_OUT / "reference_labels.npy",     labels)

# Non-zero mean confirms weights loaded and preprocess_input ran correctly
emb_mean = float(embeddings.mean())
assert emb_mean > 0.01, f"Embeddings near-zero (mean={emb_mean:.6f}) — check weight loading or preprocessing"

print(f"Saved embeddings: shape={embeddings.shape}  dtype={embeddings.dtype}  mean={emb_mean:.4f}")
print(f"Saved labels:     shape={labels.shape}  unique={set(labels.tolist())}")
print(f"  First 100 = bird (0), last 100 = drone (1)")

In [ ]:
# Cell 11 — Save eda_stats.json
#
# All values are read from computed variables — never hardcoded.
# If dataset counts change, rerun Cell 5 and re-run this cell; values update automatically.
# Schema is fixed; values are live.
#
# Downstream consumers:
#   Section 3 (data_loader.py)       → augmentation_config, class_weight, normalization_decision
#   Section 4 (transfer_learning.py) → augmentation_config, embedding_preprocessing
#   Section 6 (drift_detector.py)    → drift_baseline (file paths + shape)

stats = {
    "class_distribution": {
        split: {label: len(get_paths(split, label)) for label in LABELS}
        for split in SPLITS
    },
    "total_images": sum(
        len(get_paths(s, l)) for s in SPLITS for l in LABELS
    ),
    "class_imbalance_ratio": round(ratio, 4),
    "imbalance_decision":    imbalance_decision,
    "pixel_stats": {
        "sample_size": PIXEL_SAMPLE,
        "mean_rgb":    [round(v, 4) for v in mean_rgb.tolist()],
        "std_rgb":     [round(v, 4) for v in std_rgb.tolist()],
    },
    "imagenet_stats": {
        "mean_rgb": [0.485, 0.456, 0.406],
        "std_rgb":  [0.229, 0.224, 0.225],
    },
    "normalization_decision":  "divide_by_255_only_for_custom_cnn",
    "embedding_preprocessing": "efficientnet_preprocess_input",
    "augmentation_config":     AUGMENTATION_CONFIG,
    "class_weight": {
        "0": round(class_weight[0], 4),
        "1": round(class_weight[1], 4),
    },
    "drift_baseline": {
        "embeddings_file": "data/eda/reference_embeddings.npy",
        "labels_file":     "data/eda/reference_labels.npy",
        "shape":           list(embeddings.shape),
        "composition":     "100 bird (label=0) + 100 drone (label=1), training split",
        "label_order":     "first 100 rows = bird, last 100 rows = drone",
        "preprocessing":   "efficientnet_preprocess_input",
    },
    "generated_at": datetime.now(timezone.utc).isoformat(),
}

with open(EDA_OUT / "eda_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print("Saved: data/eda/eda_stats.json")
print(f"  class_weight:       {stats['class_weight']}")
print(f"  mean_rgb:           {stats['pixel_stats']['mean_rgb']}")
print(f"  augmentation keys:  {list(stats['augmentation_config'].keys())}")
print(f"  total_images:       {stats['total_images']}")

# Confirm augmentation config survives JSON round-trip — Section 3 reads these keys directly
with open(EDA_OUT / "eda_stats.json") as f:
    _check = json.load(f)
assert _check["augmentation_config"]["rotation_range"]     == 20,        "rotation_range mismatch"
assert _check["augmentation_config"]["height_shift_range"] == 0.1,       "height_shift_range mismatch"
assert _check["augmentation_config"]["fill_mode"]          == "nearest",  "fill_mode mismatch"
print("\nAugmentation config round-trip check passed.")

# EDA Summary — Key Findings and Decisions

## Dataset Health
- Total images: 3,319 across train / valid / test
- Class imbalance ratio: ~1.134 (mild — below 1.5 threshold)
- No corrupted images found during loading

## Decisions Locked (consumed by downstream sections)

| Decision | Value | Used By |
|---|---|---|
| Normalization (Custom CNN) | divide by 255.0 | Section 3 |
| Normalization (EfficientNetB0) | `preprocess_input()` | Section 4, drift baseline |
| Class weights | `{0: 0.941, 1: 1.066}` (sklearn balanced) | Section 3, 4 |
| Augmentation config | 6 params — see `eda_stats.json` | Section 3, 4 |
| Drift baseline | `(200, 1280)` embeddings + labels | Section 6 |

## Single Source of Truth
`data/eda/eda_stats.json` is the authority for `augmentation_config`, `class_weight`, and `normalization_decision`.  
**Never hardcode these values in training files.** Section 3 and 4 must read them from this file.

## Key Architecture Note
Drift baseline embeddings use `preprocess_input()` — not `/255.0`. This ensures the baseline and inference embeddings come from the same preprocessing pipeline. If they diverged, the KS drift score would always be non-zero regardless of actual drift.